# Question 1

1.1 Load the data frame into Python and preprocess the datasets by removing all irrelevant information using text
denoising techniques

In [2]:
import pandas as pd
import re

# load the data
df = pd.read_json("research_abstracts.json")

display(df.head())

print("\nDataset info")
df.info()

# check missing values
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicats:")
print(df.duplicated().sum())

print("\nLabel counts:")
print(df["Label"].value_counts())

import nltk
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()

# clean and reduce the words to their base form
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words).strip()

df["Clean_Abstract"] = df["Abstract"].apply(clean_text)

print("\nOriginal and cleaned text:")
display(df[["Abstract", "Clean_Abstract", "Label"]].head())

print("\nShape of the dataset:", df.shape)

,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical



Dataset info
<class 'pandas.core.frame.DataFrame'>
Index: 1138 entries, 0 to 1137
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DOI       1134 non-null   object
 1   Abstract  1138 non-null   object
 2   Label     1138 non-null   object
dtypes: object(3)
memory usage: 35.6+ KB

Missing values:
DOI         4
Abstract    0
Label       0
dtype: int64

Duplicats:
2

Label counts:
Label
classical    900
ai           238
Name: count, dtype: int64


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\xikon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Original and cleaned text:


,Abstract,Clean_Abstract,Label
0,Forward-looking forecasting of the inflation r...,forward looking forecasting of the inflation r...,ai
1,Our study examined the disaggregation of infla...,our study examined the disaggregation of infla...,ai
2,This paper develops a dynamic factor model tha...,this paper develops a dynamic factor model tha...,classical
3,"In this paper, I use a simple SIR Macro model ...",in this paper i use a simple sir macro model t...,classical
4,BackgroundGame theory is useful for identifyin...,backgroundgame theory is useful for identifyin...,classical



Shape of the dataset: (1138, 4)


1.2 Identify data columns relevant for feature engineering using feature modelling techniques.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# turn the cleaned abstracts into tf idf features
tfidf = TfidfVectorizer(stop_words="english")

X = tfidf.fit_transform(df["Clean_Abstract"])
y = df["Label"]

print("Shape of tf idF data:", X.shape)
print("Number of labels:", len(y))

Shape of tf idF data: (1138, 8268)
Number of labels: 1138


In [5]:
# check a few of the words used as features
print("\nSome of the features:")
print(tfidf.get_feature_names_out()[:20])


Some of the features:
['aa' 'aaa' 'abandoned' 'abandonment' 'abar' 'abbreviation' 'abct'
 'ability' 'able' 'abm' 'abnormal' 'abo' 'abortive' 'abroad' 'abrupt'
 'absence' 'absent' 'absenteeism' 'absolut' 'absolute']


In [6]:
from sklearn.model_selection import train_test_split

# split the data into training and testing sets
# stratify keeps the class proportions similar in both sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print("train data:", X_train.shape)
print("Test Data:", X_test.shape)

# check that both sets still have the two classes
print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

train data: (910, 8268)
Test Data: (228, 8268)

Training labels:
Label
classical    720
ai           190
Name: count, dtype: int64

Testing labels:
Label
classical    180
ai            48
Name: count, dtype: int64


1.3 Build a baseline model, provide a performance report for model learning and generalization and comment on the
performance of your model

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# build the baseline model
baseline_model = MultinomialNB()

baseline_model.fit(X_train, y_train)

# predictions for both training and testing data
y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)

# training performance
print("Training accuracy:", accuracy_score(y_train, y_train_pred))

print("\nTraining classification report:")
print(classification_report(y_train, y_train_pred))

print("\nTraining confusion matrix:")
print(confusion_matrix(y_train, y_train_pred))

# testing performance
print("\nTest accuracy:", accuracy_score(y_test, y_test_pred))

print("\nTest classification report:")
print(classification_report(y_test, y_test_pred))

print("\nTest confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

Training accuracy: 0.8098901098901099

Training classification report:
              precision    recall  f1-score   support

          ai       1.00      0.09      0.16       190
   classical       0.81      1.00      0.89       720

    accuracy                           0.81       910
   macro avg       0.90      0.54      0.53       910
weighted avg       0.85      0.81      0.74       910


Training confusion matrix:
[[ 17 173]
 [  0 720]]

Test accuracy: 0.793859649122807

Test classification report:
              precision    recall  f1-score   support

          ai       1.00      0.02      0.04        48
   classical       0.79      1.00      0.88       180

    accuracy                           0.79       228
   macro avg       0.90      0.51      0.46       228
weighted avg       0.84      0.79      0.71       228


Test confusion matrix:
[[  1  47]
 [  0 180]]
